# 课后练习解答（03.04_training_code_structure）

本解答对应《训练代码结构解析》课后练习，共 15 题。


### 问题1（单选题）

**题目：** `YoloTxtDataset` 最主要的职责是？

A. 从 VOC/YOLO 目录读取图片和 txt 标注并组成训练样本
B. 调用 ATC 转 OM
C. 执行 Git push
D. 生成 MindStudio 报告

**解答：** A

**解析：** Dataset 是训练数据进入 PyTorch DataLoader 的入口。


### 问题2（单选题）

**题目：** `collate_yolo` 的作用更接近哪一项？

A. 把一个 batch 中不同数量的目标框整理成可训练结构
B. 压缩 checkpoint
C. 修改学习率配置文件
D. 创建 Pull Request

**解答：** A

**解析：** 目标检测样本中每张图目标数量不同，需要自定义 collate。


### 问题3（单选题）

**题目：** `TinyYolo` 在本实验中的定位是？

A. 教学用轻量检测模型
B. 完整 YOLOv5 官方大模型
C. 数据下载器
D. Git LFS 客户端

**解答：** A

**解析：** 课程代码用轻量模型帮助学生跑通训练流程和调优方法。


### 问题4（单选题）

**题目：** `validate` 函数通常用于？

A. 训练后或训练间隔评估模型输出/损失
B. 转换 XML 文件编码
C. 设置 Git 用户名
D. 安装 CANN

**解答：** A

**解析：** validate 负责验证集评估，是训练闭环的一部分。


### 问题5（多选题）

**题目：** 训练主流程通常包含哪些步骤？

A. 读取配置
B. 构建 Dataset/DataLoader
C. 创建模型和优化器
D. 训练、验证并保存 checkpoint

**解答：** A、B、C、D

**解析：** 这些步骤共同构成完整训练闭环。


### 问题6（多选题）

**题目：** 代码中为 DDP 扩展预留的常见设计包括哪些？

A. rank/local_rank 参数
B. DistributedSampler
C. rank0 日志与保存
D. HCCL 后端初始化入口

**解答：** A、B、C、D

**解析：** 这些设计可以让单卡代码更容易扩展到多卡。


### 问题7（多选题）

**题目：** 目标检测训练中，一个样本通常至少需要哪些信息？

A. 图像张量
B. 目标类别
C. 目标框坐标
D. 训练 split 归属

**解答：** A、B、C、D

**解析：** 图像、类别、框和划分信息共同决定样本如何参与训练和验证。


### 问题8（判断题）

**题目：** 训练代码结构清晰的好处之一，是后续定位 loss 异常或数据加载瓶颈更容易。

**解答：** 正确

**解析：** 模块职责清楚时，可以分别检查数据、模型、损失、优化器和日志。


### 问题9（判断题）

**题目：** 只要模型能 forward，就不需要验证 DataLoader 输出的 shape 和 dtype。

**解答：** 错误

**解析：** shape/dtype 错误可能导致训练质量异常，即使 forward 偶尔能通过也不代表数据正确。


### 问题10（填空题）

**题目：** PyTorch 中负责按 batch 迭代数据的常用组件是 `____`。

**解答：** DataLoader

**解析：** DataLoader 将 Dataset 包装成可迭代的 batch 数据流。


### 问题11（填空题）

**题目：** 为了避免多进程重复保存 checkpoint，通常只在 `rank == ____` 时写文件。

**解答：** 0

**解析：** rank 0 通常作为主进程负责日志和保存。


### 问题12（简答题）

**题目：** 为什么目标检测任务需要自定义 `collate_fn`？

**解答：** 因为每张图片中的目标数量不同，普通默认 collate 很难把不同长度的标注直接堆叠成规则张量。自定义 collate 可以保留列表结构或补充 batch 索引。

**解析：** 这是检测任务和分类任务数据加载的一大差异。


### 问题13（简答题）

**题目：** 训练脚本中配置文件相比硬编码参数有什么优势？

**解答：** 配置文件能集中管理数据路径、类别数、batch size、学习率、AMP 和 warmup 等参数，便于复现实验、对比调参和迁移环境。

**解析：** 教学实验中配置化能显著降低改代码带来的错误。


### 问题14（简答题）

**题目：** 如果 loss 一直不变，应该从代码结构的哪些模块开始排查？

**解答：** 先检查 Dataset/label 是否正确，再检查模型输出 shape、loss 输入、优化器是否 step、学习率是否合理，最后检查 AMP 缩放和梯度是否异常。

**解析：** 按数据流顺序排查通常效率最高。


### 问题15（代码设计题）

**题目：** 写一段伪代码，说明训练循环中 forward、loss、backward、step 的基本顺序。

**解答：** ```python
for images, targets in train_loader:
    images = images.to(device)
    optimizer.zero_grad()
    outputs = model(images)
    loss = yolo_loss(outputs, targets)
    loss.backward()
    optimizer.step()
```

**解析：** AMP 场景会加入 autocast 和 GradScaler，但核心顺序仍是前向、损失、反向、更新。
